In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

import cosmic.utils as cu

# LSDR9x10 grizW1W2

### Add LSDR9x10 photometry

In [2]:
path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Loc.fits'  
df_raw = cu.readfile(path)

df = df_raw.copy()

cols_to_rename = {'ra_1': 'ra', 'dec_1': 'dec'}
df = df.rename(columns=cols_to_rename)

cols = ['ra', 'dec', 'z', 'zErr', 
        'DESI_TARGETID', 'DESI_RA', 'DESI_DEC', 'DESI_Z', 'DESI_ZERR',
        'DESI_DELTACHI2', 'DESI_DESI_TARGET', 'DESI_BGS_TARGET', 'DESI_PRIORITY', 
        'SDSS_specobjid', 'SDSS_ra', 'SDSS_dec', 'SDSS_z', 'SDSS_zErr', 'uid']
df = df[cols]

In [3]:
def phot_spec_cm(df):
    df = df.set_index('uid')
    dir_path = '/home/tiandc/Data/LegacySurveys/DR9x10/raw_seg_uid'
    
    cols = [
        'z_phot_mean_grz','z_phot_median_grz', 'z_phot_std_grz', 
        'z_phot_mean_i', 'z_phot_median_i', 'z_phot_std_i', 'photo_z_source', 
        'dered_mag_g', 'dered_mag_r', 'dered_mag_i', 'dered_mag_z', 
        'dered_mag_w1', 'dered_mag_w2', 'dered_mag_w3', 'dered_mag_w4', 
        'snr_g', 'snr_r', 'snr_i', 'snr_z', 
        'snr_w1', 'snr_w2', 'snr_w3', 'snr_w4', 
        'fracflux_g', 'fracflux_r', 'fracflux_i', 'fracflux_z', 
        'fracmasked_g', 'fracmasked_r', 'fracmasked_i', 'fracmasked_z', 
        'fracin_g', 'fracin_r', 'fracin_i', 'fracin_z', 
        'allmask_g', 'allmask_r', 'allmask_i', 'allmask_z',
    ]

    for i in range(72):
        path = os.path.join(dir_path, f'lsdr9x10_gal_seg_{i:02d}.fits')
        backgal_df = cu.readfile(path).set_index('uid')

        common_idx = backgal_df.index.intersection(df.index)
        df.loc[common_idx, cols] = backgal_df.loc[common_idx, cols].values

        print(f'File {i:02d} processed. Updated {len(common_idx)} records.')

    return df

df_save = phot_spec_cm(df).reset_index().apply(pd.to_numeric, errors='coerce')

path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot.fits'
cu.savefile(df_save, path)

File 00 processed. Updated 129105 records.
File 01 processed. Updated 139528 records.
File 02 processed. Updated 148944 records.
File 03 processed. Updated 158067 records.
File 04 processed. Updated 177654 records.
File 05 processed. Updated 159532 records.
File 06 processed. Updated 180106 records.
File 07 processed. Updated 151211 records.
File 08 processed. Updated 100309 records.
File 09 processed. Updated 75259 records.
File 10 processed. Updated 56676 records.
File 11 processed. Updated 71495 records.
File 12 processed. Updated 84781 records.
File 13 processed. Updated 92366 records.
File 14 processed. Updated 51364 records.
File 15 processed. Updated 1371 records.
File 16 processed. Updated 21 records.
File 17 processed. Updated 0 records.
File 18 processed. Updated 6811 records.
File 19 processed. Updated 22820 records.
File 20 processed. Updated 35768 records.
File 21 processed. Updated 64716 records.
File 22 processed. Updated 121443 records.
File 23 processed. Updated 167641

In [13]:
def snr_to_mag_err(snr):
    return 1.0857 / snr 

path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot.fits'
df_raw = cu.readfile(path)
df = df_raw.copy()

df['mag_g_Err'] = snr_to_mag_err(df['snr_g'])
df['mag_r_Err'] = snr_to_mag_err(df['snr_r'])
df['mag_i_Err'] = snr_to_mag_err(df['snr_i'])
df['mag_z_Err'] = snr_to_mag_err(df['snr_z'])
df['mag_w1_Err'] = snr_to_mag_err(df['snr_w1'])
df['mag_w2_Err'] = snr_to_mag_err(df['snr_w2'])

df['g_r'] = df['dered_mag_g'] - df['dered_mag_r']
df['r_i'] = df['dered_mag_r'] - df['dered_mag_i']
df['i_z'] = df['dered_mag_i'] - df['dered_mag_z']
df['z_w1'] = df['dered_mag_z'] - df['dered_mag_w1']
df['w1_w2'] = df['dered_mag_w1'] - df['dered_mag_w2']

# 星等大于0
mask = df[['dered_mag_g', 'dered_mag_r', # 这里先不区分i波段
           'dered_mag_z', 'dered_mag_w1', 'dered_mag_w2']].gt(0).all(axis=1)
df = df[mask]
print('After mag > 0:', len(df))

# 星等保证正常
mask = df[['dered_mag_g', 'dered_mag_r', # 这里先不区分i波段
           'dered_mag_z', 'dered_mag_w1', 'dered_mag_w2']].lt(30).all(axis=1)
df = df[mask]
print('After mag < 30:', len(df))

# 清理frac
def clean_FRAC(df):
    idx = (df['fracflux_g'] < 0.5) & (df['fracflux_r'] < 0.5) & (df['fracflux_z'] < 0.5)
    idx &= (df['fracmasked_g'] < 0.4) & (df['fracmasked_r'] < 0.4) & (df['fracmasked_z'] < 0.4)
    idx &= (df['fracin_g'] > 0.3) & (df['fracin_r'] > 0.3) & (df['fracin_z'] > 0.3)
    return df[idx]
df = clean_FRAC(df)
print('After clean FRAC:', len(df))

# 限制snr
def clean_snr(df):
    idx = (df.snr_g > 5) & (df.snr_r > 5) & (df.snr_z > 5)
    idx &= (df.snr_w1 > 5) & (df.snr_w2 > 5)
    return df[idx]
df = clean_snr(df)
print('After clean snr:', len(df))

output_path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean.fits'
cu.savefile(df, output_path)

After mag > 0: 9905001
After mag < 30: 9843028
After clean FRAC: 9455834
After clean snr: 7922305


- grizW1W2

In [17]:
path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean.fits'
df_raw = cu.readfile(path)

mask = (df_raw['dered_mag_i'] > 0) & (df_raw['dered_mag_i'] < 30)
mask &= (df_raw['z'] > 0.) & (df_raw['z'] <= 2.)
df_i = df_raw[mask]
print('grizW1W2:', len(df_i))

# split into train, val, test with 8:1:1 ratio
from sklearn.model_selection import train_test_split
df_trainval, df_test = train_test_split(df_i, test_size=0.1, random_state=42, shuffle=True)
df_train, df_val = train_test_split(df_trainval, test_size=0.1111, random_state=42, shuffle=True)

print(f"train: {len(df_train)}, val: {len(df_val)}, test: {len(df_test)}")

path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_i_train.fits'
cu.savefile(df_train, path)
path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_i_val.fits'
cu.savefile(df_val, path)
path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_i_test.fits'
cu.savefile(df_test, path)

grizW1W2: 3963602
train: 3170920, val: 396321, test: 396361


- grzW1W2

In [19]:
path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean.fits'
df_raw = cu.readfile(path)

mask = (df_raw['z'] > 0.) & (df_raw['z'] <= 2.)
df = df_raw[mask]
print('grzW1W2:', len(df))

# split into train, val, test with 8:1:1 ratio
from sklearn.model_selection import train_test_split
df_trainval, df_test = train_test_split(df, test_size=0.1, random_state=42, shuffle=True)
df_train, df_val = train_test_split(df_trainval, test_size=0.1111, random_state=42, shuffle=True)

print(f"train: {len(df_train)}, val: {len(df_val)}, test: {len(df_test)}")

path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_train.fits'
cu.savefile(df_train, path)
path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_val.fits'
cu.savefile(df_val, path)
path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_test.fits'
cu.savefile(df_test, path)

grzW1W2: 7921838
train: 6337549, val: 792105, test: 792184


### 使用grizW1W2 训练集划分SDSS/DESI后重新分别划分训练集和验证集

In [ ]:
path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_i_train.fits'
df_raw = cu.readfile(path)

# DESI only
desi_df = df_raw[df_raw['DESI_Z']>0]
print('DESI training sample size:', desi_df.shape)
from sklearn.model_selection import train_test_split
df_train, df_val = train_test_split(desi_df, test_size=0.1, random_state=42, shuffle=True)
print(f"DESI only:  train: {len(df_train)}, val: {len(df_val)}")

path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_i_train_DESIonly_train.fits'
cu.savefile(df_train, path)
path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_i_train_DESIonly_val.fits'
cu.savefile(df_val, path)

# SDSS only
sdss_df = df_raw[df_raw['SDSS_z']>0]
print('SDSS training sample size:', sdss_df.shape)
from sklearn.model_selection import train_test_split
df_train, df_val = train_test_split(sdss_df, test_size=0.1, random_state=42, shuffle=True)
print(f"SDSS only:  train: {len(df_train)}, val: {len(df_val)}")

path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_i_train_SDSSonly_train.fits'
cu.savefile(df_train, path)
path = './data/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_i_train_SDSSonly_val.fits'
cu.savefile(df_val, path)

DESI training sample size: (2617506, 69)
DESI only:  train: 2355755, val: 261751
SDSS training sample size: (758864, 69)
SDSS only:  train: 682977, val: 75887


In [62]:
df_desi.columns

Index(['uid', 'ra', 'dec', 'z', 'zErr', 'DESI_TARGETID', 'DESI_RA', 'DESI_DEC',
       'DESI_Z', 'DESI_ZERR', 'DESI_DELTACHI2', 'DESI_DESI_TARGET',
       'DESI_BGS_TARGET', 'DESI_PRIORITY', 'SDSS_specobjid', 'SDSS_ra',
       'SDSS_dec', 'SDSS_z', 'SDSS_zErr', 'z_phot_mean_grz',
       'z_phot_median_grz', 'z_phot_std_grz', 'z_phot_mean_i',
       'z_phot_median_i', 'z_phot_std_i', 'photo_z_source', 'dered_mag_g',
       'dered_mag_r', 'dered_mag_i', 'dered_mag_z', 'dered_mag_w1',
       'dered_mag_w2', 'dered_mag_w3', 'dered_mag_w4', 'snr_g', 'snr_r',
       'snr_i', 'snr_z', 'snr_w1', 'snr_w2', 'snr_w3', 'snr_w4', 'fracflux_g',
       'fracflux_r', 'fracflux_i', 'fracflux_z', 'fracmasked_g',
       'fracmasked_r', 'fracmasked_i', 'fracmasked_z', 'fracin_g', 'fracin_r',
       'fracin_i', 'fracin_z', 'allmask_g', 'allmask_r', 'allmask_i',
       'allmask_z', 'mag_g_Err', 'mag_r_Err', 'mag_i_Err', 'mag_z_Err',
       'mag_w1_Err', 'mag_w2_Err', 'g_r', 'r_i', 'i_z', 'z_w1', 'w1_w2'],
 

# LSDR9x10 grzW1W2 + PS1DR2 i

In [ ]:
# Add LSDR9x10xPS1DR2 photometry
path = './data/LSDR9x10xPS1DR2/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_xPS1DR2Loc.fits'  
df_raw = cu.readfile(path)

df = df_raw.copy()

cols_to_rename = {'ra_1': 'ra', 'dec_1': 'dec'}
df = df.rename(columns=cols_to_rename)

In [35]:
def phot_spec_cm(df):
    df = df.set_index('objID')
    dir_path = '/home/tiandc/Data/PanSTARRS/DR2/final'
    
    cols = [
        'iPSFMag_dered', 'iKronMag_dered', 'iApMag_dered', 
        'iPSFMagErr', 'iApMagErr', 'iKronMagErr',
    ]

    for i in range(72):
        path = os.path.join(dir_path, f'ps1dr2_final_{i:d}.fits')
        backgal_df = cu.readfile(path).set_index('objID')

        common_idx = backgal_df.index.intersection(df.index)
        df.loc[common_idx, cols] = backgal_df.loc[common_idx, cols].values

        print(f'File {i:02d} processed. Updated {len(common_idx)} records.')

    return df

df_save = phot_spec_cm(df).reset_index().apply(pd.to_numeric, errors='coerce')

path = './data/LSDR9x10xPS1DR2/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_xPS1DR2Photi.fits'
cu.savefile(df_save, path)

File 00 processed. Updated 75744 records.
File 01 processed. Updated 78846 records.
File 02 processed. Updated 88348 records.
File 03 processed. Updated 88321 records.
File 04 processed. Updated 99729 records.
File 05 processed. Updated 89211 records.
File 06 processed. Updated 98273 records.
File 07 processed. Updated 77541 records.
File 08 processed. Updated 48967 records.
File 09 processed. Updated 43152 records.
File 10 processed. Updated 32631 records.
File 11 processed. Updated 45835 records.
File 12 processed. Updated 51557 records.
File 13 processed. Updated 52963 records.
File 14 processed. Updated 24876 records.
File 15 processed. Updated 550 records.
File 16 processed. Updated 0 records.
File 17 processed. Updated 0 records.
File 18 processed. Updated 4242 records.
File 19 processed. Updated 16627 records.
File 20 processed. Updated 28731 records.
File 21 processed. Updated 49416 records.
File 22 processed. Updated 84216 records.
File 23 processed. Updated 102217 records.
Fi

- 用所有样本的grz

In [41]:
path = './data/LSDR9x10xPS1DR2/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_xPS1DR2Photi.fits'
df_raw = cu.readfile(path)

idx = (df_raw['z'] > 0.) & (df_raw['z'] <= 2.)
df_allgrz = df_raw[idx]

# split into train, val, test with 8:1:1 ratio
from sklearn.model_selection import train_test_split
df_trainval, df_test = train_test_split(df_allgrz, test_size=0.1, random_state=42, shuffle=True)
df_train, df_val = train_test_split(df_trainval, test_size=0.1111, random_state=42, shuffle=True)

print(f"train: {len(df_train)}, val: {len(df_val)}, test: {len(df_test)}")

path = './data/LSDR9x10xPS1DR2/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_xPS1DR2Photi_train.fits'
cu.savefile(df_train, path)
path = './data/LSDR9x10xPS1DR2/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_xPS1DR2Photi_val.fits'
cu.savefile(df_val, path)
path = './data/LSDR9x10xPS1DR2/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_xPS1DR2Photi_test.fits'
cu.savefile(df_test, path)

train: 5126936, val: 640795, test: 640860


- 只使用没有i波段数据的样本

In [53]:
path = './data/LSDR9x10xPS1DR2/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_xPS1DR2Photi.fits'
df_raw = cu.readfile(path)

idx = (df_raw['z'] > 0.) & (df_raw['z'] <= 2.)
idx &= np.isinf(df_raw['dered_mag_i']) | (df_raw['dered_mag_i'] == -9999)
df_grz = df_raw[idx]

# split into train, val, test with 8:1:1 ratio
from sklearn.model_selection import train_test_split
df_trainval, df_test = train_test_split(df_grz, test_size=0.1, random_state=42, shuffle=True)
df_train, df_val = train_test_split(df_trainval, test_size=0.1111, random_state=42, shuffle=True)

print(f"train: {len(df_train)}, val: {len(df_val)}, test: {len(df_test)}")

path = './data/LSDR9x10xPS1DR2/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_xPS1DR2Photi_grz_train.fits'
cu.savefile(df_train, path)
path = './data/LSDR9x10xPS1DR2/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_xPS1DR2Photi_grz_val.fits'
cu.savefile(df_val, path)
path = './data/LSDR9x10xPS1DR2/DESIDR1_xSDSSDR19_xlsdr9x10Phot_clean_xPS1DR2Photi_grz_test.fits'
cu.savefile(df_test, path)

train: 2636402, val: 329514, test: 329547


In [52]:
df_grz.groupby(df_grz['dered_mag_i']).size()

dered_mag_i
-9999.0    1936604
 inf       1358859
dtype: int64